# CDI Layer 2 and Layer 3

Use the **compute** kernel. After repository code changes, restart the kernel once. Put `OPENAI_API_KEY` in the repository `.env`, replace `FACT_SHEET` below with an absolute path to a structured Markdown fact sheet, and run this cell. It creates a resumable eight-mission Layer 2 run and shows the deterministic report plus one mission for inspection.

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.cli import load_dotenv_key, run_all as run_layer2
from ML.deep_research.layer2.create_run import create_run as create_layer2_run
from ML.deep_research.layer2.fs import load_json, read_text
from ML.deep_research.layer2.settings import PLANNER_PATH, RUNS_DIR

FACT_SHEET = Path(r"fact_sheet.md")
PREVIEW_MISSION = 0

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 2.")

L2_RUN = create_layer2_run(FACT_SHEET, PLANNER_PATH, RUNS_DIR)
print(f"Layer 2 run created: {L2_RUN}")
print(f"Resume if interrupted: .\\run.ps1 -Resume '{L2_RUN}'")
L2_CHECKS = await asyncio.to_thread(run_layer2, L2_RUN)

l2_record = load_json(L2_RUN / "run.json")
mission_files = sorted((L2_RUN / "missions").glob("*.json"))
display(Markdown(read_text(L2_RUN / "check_report.md")))
display(Markdown("### Recorded Layer 2 usage"))
display(JSON(data=l2_record.get("usage", {}), expanded=True))
print(f"Mission files: {len(mission_files)}")
for path in mission_files:
    print(f"- {path.name}")
if mission_files:
    preview = mission_files[PREVIEW_MISSION]
    display(Markdown(f"### Mission preview: `{preview.name}`"))
    display(JSON(data=load_json(preview), expanded=False))


## Layer 3 — live online research

Run this only after Layer 2 completes. **This cell confirms that the input is public or invented, sends research queries to external services, and consumes model/web-search usage.** Layer 3 currently uses low reasoning for measured live testing; there is no separate test-mode command. Eight researchers run in parallel, followed by one review, one optional clarification batch, and synthesis. The cell shows every published domain report, the review, usage, and the final property answer.

In [1]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.cli import load_dotenv_key
from ML.deep_research.layer2.fs import load_json, read_text
from ML.deep_research.layer3.cli import run_all as run_layer3
from ML.deep_research.layer3.pipeline.create_run import create_run as create_layer3_run
from ML.deep_research.layer3.settings import RUNS_DIR as LAYER3_RUNS_DIR, SCHEMA_VERSION
from ML.deep_research.layer3.usage import summarize_usage

PUBLIC_INPUT_CONFIRMED = True
load_dotenv_key()
if not globals().get("L2_RUN"):
    for candidate in sorted(
        LAYER3_RUNS_DIR.glob("L2_*"), key=lambda path: path.stat().st_mtime, reverse=True
    ):
        record = load_json(candidate / "run.json")
        checks = record.get("checks", {})
        if checks.get("run") and checks.get("passed") == checks.get("run"):
            L2_RUN = candidate
            break
    else:
        raise RuntimeError("No completed Layer 2 run was found under runs/.")
L2_RUN = Path(L2_RUN)
print(f"Using existing Layer 2 run: {L2_RUN}")

if not PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 3 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 3.")

for candidate in sorted(
    LAYER3_RUNS_DIR.glob("L3_*"), key=lambda path: path.stat().st_mtime, reverse=True
):
    candidate_record = load_json(candidate / "run.json")
    source_path = candidate_record.get("source_l2", {}).get("path", "")
    if (
        candidate_record.get("schema_version") == SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L2_RUN.resolve()
    ):
        L3_RUN = candidate
        break
else:
    L3_RUN = create_layer3_run(
        L2_RUN, LAYER3_RUNS_DIR, public_input_confirmed=PUBLIC_INPUT_CONFIRMED
    )

l3_before = load_json(L3_RUN / "run.json")
execution = l3_before.get("execution", {})
records = [
    *execution.get("domains", {}).values(),
    execution.get("review", {}),
    execution.get("final", {}),
    *(((execution.get("clarification") or {}).get("domains", {})).values()),
]
retry_failed = any(item.get("status") == "failed" for item in records)
retry_flag = " -RetryFailed" if retry_failed else ""
resume_command = f".\\run.ps1 -ResumeL3 '{L3_RUN}'{retry_flag}"
print(f"Layer 3 run: {L3_RUN}")
print(f"Resume if interrupted: {resume_command}")
layer3_task = None
if l3_before.get("status") != "verified":
    layer3_task = asyncio.create_task(run_layer3(L3_RUN, retry_failed=retry_failed))
while layer3_task and not layer3_task.done():
    await asyncio.sleep(5)
    live = load_json(L3_RUN / "run.json")
    lines = read_text(L3_RUN / "usage.jsonl").splitlines()
    latest = json.loads(lines[-1]) if lines else {}
    running = [
        name for name, item in live.get("execution", {}).get("domains", {}).items()
        if item.get("status") == "running"
    ]
    partials = sorted((L3_RUN / "domains").glob("*.partial.md"))
    clear_output(wait=True)
    print(f"Layer 3 run: {L3_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Running domains: {len(running)}")
    for name in running:
        print(f"- {name}")
    print(f"Progressive reports: {len(partials)}/8")
    print("Usage:", summarize_usage(L3_RUN))
    if latest:
        print("Last activity:", latest.get("timestamp"), latest.get("actor"), latest.get("phase"), latest.get("detail", ""))
L3_CHECKS = await layer3_task if layer3_task else []
clear_output(wait=True)

l3_record = load_json(L3_RUN / "run.json")
domain_files = sorted(
    path for path in (L3_RUN / "domains").glob("*.md")
    if not path.name.endswith(".partial.md")
)
partial_files = sorted((L3_RUN / "domains").glob("*.partial.md"))
review = L3_RUN / "review" / "final_review.md"
final_answer = L3_RUN / "research" / "final.md"
display(Markdown(read_text(L3_RUN / "check_report.md")))
display(Markdown("### Recorded Layer 3 usage"))
display(JSON(data=l3_record.get("usage", {}), expanded=True))
print(f"Domain reports: {len(domain_files)}")
for path in domain_files:
    print(f"- {path.name}")
print(f"Progressive report logs retained: {len(partial_files)}")
display(Markdown("### Final review"))
display(Markdown(read_text(review)))
display(Markdown("### Property synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; see the check report above."))


# Run L3_20260821_8c04 — Layer 3 check report

14 checks · 13 passed · 1 failed

## Checks

- [x] 1. The recorded Layer 2 result passed every check
- [x] 2. Skill and prompt snapshots match their recorded hashes
- [x] 3. Run schema, harness, model policy, provider, and public-input record are valid
- [x] 4. Eight domains, one review, optional single clarification, and synthesis are complete
- [x] 5. All 8 published and progressive domain reports are non-empty — expected=8 published and 8 partial
- [x] 6. The adversarial review is non-empty
- [x] 7. The final property answer is non-empty
- [x] 8. Every retained raw source hashes to its filename
- [x] 9. Source index paths and hashes resolve
- [x] 10. Query records are unique and structurally valid
- [x] 11. Citation records are unique and exact quotations revalidate
- [ ] 12. Every emitted citation marker resolves to verified evidence
- [x] 13. An answered property has a verified final-answer citation
- [x] 14. Usage and progress events are attributable, timestamped, and unique


### Recorded Layer 3 usage

<IPython.core.display.JSON object>

Domain reports: 8
- asset-integrity-systems-and-operational-resilience.md
- energy-carbon-and-transition.md
- external-dependencies-geopolitics-trade-and-supply-chains.md
- finance-debt-and-macro-transmission.md
- ground-physical-climate-and-insurability.md
- location-demand-market-valuation-and-exit.md
- occupier-lease-income-and-counterparty-economics.md
- rights-public-law-and-ownership-governance.md
Progressive report logs retained: 8


### Final review

## Decision-level conclusions

**Overall decision: do not approve an unconditional acquisition, refinancing, or valuation conclusion on the present record.** The property appears to be an operating courthouse/office complex with stated annualised contractual rent of approximately **€939,062.04**, but the investment case is not decision-ready because the core drivers of value and downside are unverified: title and Hesse use rights, lease enforceability and collectability, current building and life-safety condition, maintenance continuity, garage CapEx, insurance, debt terms, and market value/liquidity.

The appropriate posture is a **conditional diligence hold** rather than a definitive reject. Conditions precedent should include:

- certified title, cadastral and land-charge reconciliation, including the Hesse easement and canal-right instrument;
- executed lease chain, indexation notices, payment ledger, security and complete cost-allocation evidence;
- current integrated technical, fire-safety, electrical, ventilation, CO, drainage, waterproofing and structural verification;
- confirmation of whether garage works are actually contracted, with fixed scope, programme, contingency, disruption plan and allocation of liability;
- replacement or novation of the non-transferring FMC maintenance arrangement;
- current insurance terms, claims history and exclusions;
- executed debt and security documents, lender consent and current payoff/balance information; and
- an independent valuation supported by current market rent, yield, buyer-depth and exit evidence.

If these conditions are not satisfied, the asset should be underwritten with a material technical/legal contingency, unquantified refinancing risk and conservative exit assumptions. The present evidence supports operational continuity as a working hypothesis, not as a confirmed low-risk conclusion.

### Contradictions and evidence supporting each reading

1. **Garage procurement versus actual commitment.** Three bids cluster around €703k–€720k net, with Chemicon recommended at €702,668.06 net against a €765,000 net estimate. This supports a project-specific indicative cost range and evidence of competitive tendering. However, the later February 2026 ptd appointment is for architectural LP8 services, not proof of a construction award. Both readings must be retained: there is likely a material refurbishment need, but neither final price, scope, programme nor purchaser liability is established.

2. **Maintenance coverage.** The FMC framework reportedly covers owner maintenance obligations, but it will not transfer on sale. Therefore the property may currently have operational support, while the buyer has no assured post-closing continuity. A novation, replacement contract or procurement process is required.

3. **Fire-safety status.** Dated fire-door reports record defects, while January 2026 records missing HPPVO/LBIH evidence and outdated escape/rescue plans. These may reflect documentary-transfer failure or already-completed remediation, but no current acceptance or close-out evidence is supplied. Historic design requirements—such as fire-water provision of at least 2,400 l/min for two hours—are not current compliance certification.

4. **Parking records.** The records consistently indicate 89 required spaces, but the allocation differs between approximately 58 underground plus 39 visitor spaces and 55 underground plus 34 outdoor spaces. The totals reconcile arithmetically, but the physical/legal allocation does not. The parking-replacement calculation also shows DM213,000 before reference to a DM200,000 statutory cap. Neither the operative liability nor its current euro equivalent is established.

5. **Drainage and water evidence.** Historic drainage totals differ slightly, at 33.52 versus 33.62 l/s. The difference may be immaterial, but it signals that the approved hydraulic basis and canal-right access should be reconciled. Groundwater exposure, mineral-spring/drinking-water protection and heavy-rain flow-path modelling support a real physical-risk pathway, but do not prove flooding, ingress or loss at the property.

6. **Title and debt-charge amounts.** The property file identifies PATRIZIA Hessen Eins GmbH & Co. KG, parcels Flur 8 Gonzenheim 122/11 and 123/1, and a Eurohypo land charge, but contains inconsistent parcel identifiers and land-charge amounts of €595,375,000 versus €595,375. The charge is security, not evidence of current debt. A transcription error, historic released charge, portfolio security or live encumbrance are all possible until the certified Grundbuch and loan documents are reviewed.

7. **Use and planning flexibility.** The §34 BauGB setting, absence of an identified Bebauungsplan or Veränderungssperre, and broad conditional Hesse easement wording support potential continuity and some adaptive-use optionality. They do not establish unconditional permission for alternative use, nor do they resolve water-protection, fire-safety, archaeology, title or easement conditions.

8. **Environmental status.** The April 2026 ALTIS enquiry and historic “no suspicion” evidence are supportive database/no-knowledge indications. They are not equivalent to current contamination, groundwater, ordnance or archaeological clearance. The absence of a recorded entry should not be treated as evidence that no physical risk exists.

### Shared-source dependence and apparent agreement that is not independent confirmation

Most domain conclusions rely on the same supplied property-file extracts, data-room representations and procurement records. Agreement between Asset Integrity, Occupier/Lease, Location/Valuation, Finance and External Dependencies on the FMC non-transfer, rent figures, parking inconsistency, garage bids and missing safety evidence is therefore **corroboration of consistent extraction from one source, not independent confirmation**.

The €78,255.17 monthly rent and €702,668.06 garage offer recur across domains but remain single-source facts. Likewise, the planning, easement, groundwater, fire-water and historic inspection statements are not independently validated current conditions. The genuinely external evidence is limited mainly to general statutory, municipal hazard-model, ECB, Eurostat and technical-framework material. Those sources establish context or methodology, not property-specific compliance, loss, value or financing.

### Duplicated financial effects and risks

The following effects must not be added repeatedly across domains:

- **Garage works:** the approximately €703k net base cost is one CapEx item. Asset Integrity, Location/Valuation, Finance and External Dependencies each describe its consequences, but it should be counted once, with separate contingency, fees, tax, escalation, financing carry and disruption assumptions.
- **Maintenance replacement:** non-transfer of FMC is one potential Opex/leakage item, not separate costs in lease, resilience, finance and supply-chain models.
- **Fire-safety and technical backlog:** remediation should be one technical reserve, with any resulting downtime, professional fees, insurance impact and financing carry modelled as linked consequences rather than duplicate CapEx.
- **Groundwater, drainage and climate risk:** waterproofing, drainage upgrades, insurance deductibles, business interruption and delay are related but distinct. Avoid both reserving the full physical remediation cost and separately applying an uncalibrated generic hazard discount without evidence of double counting.
- **Parking:** temporary parking, statutory replacement payment, access disruption and tenant compensation should be separated and tied to the same works programme.
- **Rent downside:** vacancy, collection loss, invalid indexation and unrecovered operating costs are alternative or overlapping income stresses. They should not be stacked at full severity unless the model explicitly represents simultaneous occurrence.
- **Debt and valuation:** rate stress, yield expansion and lower NOI are separate transmission mechanisms, but their combined impact should flow through a single integrated cash-flow, covenant and exit model.

### Cross-domain causal chains, handoffs and broken links

1. **Title/use → income → value/finance.** Parcel and land-charge reconciliation, Hesse use rights and easements determine lawful ownership, possession and use. These feed lease enforceability, collateral validity, saleability and exit value. The chain is currently broken at certified title, easement instruments and lender/security evidence.

2. **Lease/indexation → gross income → NOI → debt service/value.** The stated CPI-linked rent increases produce €939k annualised gross rent, but payment history, tenant identity, security, recoveries, landlord costs, lease term and written-form compliance are missing. The chain is broken between stated rent and verified collectible NOI.

3. **Condition/fire safety → works → disruption → income/exit.** Open or undocumented safety items may require works, temporary controls or restricted courthouse access. Those consequences pass to lease economics, valuation and financing. The link is broken by absent current inspections, close-outs, programme and responsibility allocation.

4. **Garage tender → CapEx/parking disruption → liquidity/valuation.** The tender establishes an indicative base cost only. Scope, award, timing, parking closure, temporary arrangements and cost allocation are missing, so no reliable cash-flow or exit adjustment can yet be made.

5. **Groundwater/water protection/drainage → permits/design → cost/insurability.** Protection-zone constraints and hydraulic/easement uncertainty may affect garage or energy works, insurance and delay. The chain is broken by absent parcel-specific permissions, current hydraulic verification, intrusive testing and policy terms.

6. **Energy data → transition CapEx/Opex → value.** No meter or utility series is available. Consequently, energy cost, emissions, savings, regulatory exposure and retrofit economics cannot be connected reliably to valuation or financing. Gross volume and generic benchmarks are insufficient.

7. **External suppliers/authorities → delivery programme → CapEx and exit.** No evidenced geopolitical or trade-specific exposure is established. The immediate dependency is more basic: contractor capacity, contract continuity, municipal interfaces and approvals. The chain remains an unknown rather than a low-risk conclusion.

### Material unknowns and whether they should remain unknown

Some gaps are likely resolvable and should be treated as transaction conditions: current Grundbuch and title instruments; executed lease and notices; payment/security records; current inspections and authority close-outs; maintenance-contract transfer terms; garage award and scope; insurance schedule; debt documents; approved plans and permits; and current valuation evidence.

Other matters should remain honestly unknown until property-specific evidence exists: actual latent structural condition, future climate loss, insurance pricing, energy performance, retrofit return, counterparty resilience, refinancing terms and alternative-use value. Generic public searches are unlikely to resolve these reliably. The reports correctly avoid converting city-wide hazard models, historical design documents, database no-findings or generic benchmarks into asset-specific conclusions.

### Implications

- **Income:** Use €939,062.04 only as stated gross contractual rent, subject to verification. Do not underwrite it as NOI or durable low-risk income.
- **Opex:** Allow for replacement maintenance, unrecovered owner obligations, insurance and possible safety/operational costs. Amounts are currently unknown.
- **CapEx:** Carry the uncommitted garage refurbishment at approximately €702,668 net as an indicative base only, plus contingency, fees, taxes, escalation and disruption. Add separate reserves for technical, fire, waterproofing, drainage, energy and public-law exposure only once scopes are defined.
- **Value:** No supported market value, yield, alternative-use premium or liquidity conclusion is available. Apply sensitivities for income haircut, CapEx, downtime, yield expansion and delayed exit.
- **Financing:** Debt balance, DSCR, LTV, covenants, refinancing proceeds and lender consent are unknown. Do not rely on the Eurohypo charge amount or approve leverage from headline rent.
- **Exit:** Saleability depends on clean title, enforceable use rights, current compliance evidence, documented CapEx, insurability, tenant durability and buyer depth. Until verified, assume longer marketing, a narrower buyer pool and greater diligence discount rather than a confirmed exit impairment.

### Property synthesis

# Final Property Decision

## Decision outcome

**Do not approve an unconditional acquisition, refinancing, or valuation conclusion on the present record.** Adopt a **conditional diligence hold**.

The property appears to be an operating courthouse/office complex at Auf der Steinkaut 10–12, Bad Homburg, with stated annualised gross contractual rent of **€939,062.04**. That income, however, is not yet verified as collectible NOI, and the principal drivers of value and downside remain unresolved: title and Hesse use rights, lease enforceability, current building and life-safety condition, maintenance continuity, garage CapEx, insurance, debt terms, energy performance, and market liquidity.

The record supports operational continuity as a working hypothesis, **not** as a confirmed low-risk conclusion. Final approval should be conditional on the evidence package below. If the conditions are not met, the asset should be underwritten with a material legal/technical contingency, unquantified refinancing risk, and conservative exit assumptions.

> All researchers use the same model and shared runtime. Agreement between reports is therefore not independent confirmation, and it does not establish field-wide consensus. Much of the apparent agreement reflects consistent extraction from the same supplied property-file and data-room materials.

## Facts driving the decision

- The latest stated rent is **€78,255.17 per month**, or **€939,062.04 per year**, before vacancy, collection losses, recoveries, landlord costs, taxes, concessions, or CapEx. The documented increases support a stated CPI-linked rent history, but payment, security, tenant identity, lease term, written-form compliance, and recoverability remain unverified.
- The garage refurbishment has three close tender offers: Chemicon **€702,668.06 net**, Karrié **€704,397.29 net**, and Teixeira **€719,950.93 net**, against a **€765,000 net** estimate. This supports an indicative project-specific base cost, but there is no evidence of a construction award, final scope, programme, contingency, or purchaser liability. The February 2026 ptd appointment is for architectural LP8 services and is not a construction award.
- The FMC framework reportedly covers owner maintenance obligations but **will not transfer on sale**, creating a post-closing continuity and landlord-leakage risk.
- Current technical compliance is not established. Dated fire-door reports identify defects, while January 2026 records outdated escape/rescue plans and missing HPPVO/LBIH remediation evidence. Historic requirements, including fire-water provision of at least **2,400 l/min for two hours**, are not current compliance certification.
- Title and security records contain material inconsistencies: parcel identifiers and Eurohypo land-charge amounts are inconsistent, including **€595,375,000 versus €595,375**. A land charge is security, not proof of current debt.
- The property has groundwater, mineral-spring/drinking-water protection, drainage and heavy-rain exposure. The city hazard model includes parcel-based flow-path maps and rainfall scenarios including **80 mm in one hour**, but modelling is not evidence of property-specific loss. citef69d718c2365aae5868cb4d9fe6d934052f928fc3227b58b6adff409cdce4134 citedf7368946b7136f03ccafe9110991a4ce11c7e09722feab085d11f64ac1b1b7e
- No reliable current energy or GHG performance is available: there are no utility bills, meter map, BMS data, fuel records, operating-hour data, or control matrix.
- No independent valuation, current market-rent evidence, yield evidence, buyer-depth analysis, or comparable transaction set has been established.

## Asset and income performance

### Asset performance

The asset is documented as an operating judicial/office complex, which supports a continuity case. The built-up §34 BauGB setting, absence of an identified Bebauungsplan or Veränderungssperre, and broad conditional Hesse easement wording also suggest potential adaptive-use optionality. Those points remain **supported or inferential only**, because title, easement conditions, permits, fire-safety approvals, water-protection restrictions, archaeology, and current physical condition have not been reconciled.

The immediate asset-performance concern is not an evidenced failure event but an **incompletely evidenced backlog**:

- current structure, envelope, MEP, controls, communications, and utility condition are unknown;
- groundwater and waterproofing requirements may affect basement and garage works;
- drainage totals differ slightly at **33.52 versus 33.62 l/s** and require reconciliation;
- fire-door, detector, escape-plan, HPPVO and LBIH records are incomplete;
- garage refurbishment appears likely to require material CapEx, but the commitment and final cost are unknown;
- the maintenance framework will not transfer automatically.

The indicative garage base cost of approximately **€703,000 net** should be counted once in the integrated model, with separate allowances for contingency, professional fees, tax, escalation, financing carry, disruption, temporary parking, and security/access arrangements. It must not be duplicated across technical, finance, valuation, and external-dependency cases.

### Income performance

The documented rent trajectory is:

- before January 2023: **€61,440.27 per month**;
- January 2023: **€66,232.61**;
- January 2024: **€72,458.48**;
- January 2026: **€78,255.17**.

These figures support a positive stated rent history, subject to the lease’s reported 7.5% CPI trigger. They do **not** establish:

- valid execution and written-form compliance of the lease chain;
- correct CPI reference dates, calculations, notices, or delivery;
- current payment and arrears status;
- occupier identity, financial capacity, guarantee, or security;
- lease expiry, break, renewal, or termination rights;
- service-charge recoverability and reconciliations;
- landlord-paid costs, concessions, VAT treatment, or area changes.

Accordingly, **€939,062.04 is a stated gross contractual-rent case only**, not NOI and not a low-risk income conclusion. Vacancy, collection loss, invalid or disputed indexation, unrecovered operating costs, maintenance replacement, parking obligations, and garage disruption should be modelled as distinct but potentially correlated stresses.

## Eight domain conclusions

### 1. Asset integrity, systems and operational resilience — **Inference / unknown**

The property is operating, but current structural, envelope, MEP, fire, electrical, ventilation, CO, communications, controls, drainage, waterproofing, and utility condition is not established. Historic design requirements and dated inspection observations evidence risk pathways, not present compliance.

The garage refurbishment is an uncommitted need with an indicative base offer of **€702,668.06 net**. The key decision question is whether the works are mandatory for safety or waterproofing, and what downtime and parking restrictions they create.

**Decision consequence:** Require a current integrated condition survey, current safety and systems testing, fire-water confirmation, basement/drainage review, full remediation close-outs, and verified works scope/programme before approval.

### 2. Occupier, lease income and counterparty economics — **Supported for lease-history facts; otherwise unknown**

The lease chain and stated rent increases are documented in the supplied records, but the executed instruments, payment ledger, security, tenant identity, recoveries, cost allocation, and enforceability have not been verified. The non-transfer of FMC creates a direct potential leakage and continuity issue.

Parking requirements are consistently stated at **89 spaces**, but the underground/outdoor allocation differs between historic records: approximately **58+39** versus **55+34**. The totals reconcile arithmetically, but the physical and legal allocation does not.

**Decision consequence:** Treat rent as provisional gross contractual income. Obtain the executed lease package, notices, payment history, security, recoveries, maintenance allocation, parking agreement, and disruption arrangements.

### 3. Rights, public law and ownership governance — **Unknown**

PATRIZIA Hessen Eins GmbH & Co. KG and parcels Flur 8 Gonzenheim 122/11 and 123/1 are identified in the supplied file, but current title, parcel history, easements, land charges, corporate authority, and State of Hesse rights remain unreconciled. The Hesse-use easement may support continued or alternative use, but its exact conditions and priority are unknown.

The planning record indicates §34 BauGB, no identified Bebauungsplan, and no Veränderungssperre. This is not confirmation that current courthouse use, alterations, parking, fire systems, roof expansion, or proposed alternative uses are lawful. Water-protection and archaeological positions also remain open.

The legal route for obtaining the Grundbuch and referenced instruments is supported by the official rule that inspection is available to a person demonstrating legitimate interest, including documents referenced in the register [75e3566d976dbb473949ad171d22b5c9db709c4a9ba0926cef38c69f0025cf5c].

**Decision consequence:** Certified Grundbuch, Grundakten, cadastral reconciliation, easement instruments, charge-release evidence, corporate authority, permits, approved plans, water-protection confirmation, archaeological response, and litigation search are gating items.

### 4. Ground, physical climate and insurability — **Inference / unknown**

Groundwater, water-protection designation, drainage uncertainty, surface-water exceedance, fire impairment, contamination, ordnance, and archaeology are credible risk pathways. The city hazard model is useful context and provides parcel-based maps, but it relies on simplified assumptions and does not prove flooding or loss at the property. cite8eb489015110e563335a625d4bce8e85daaf26a5c4ab89d4818eea2bc03a4329

The water-protection designation can prohibit groundwater-endangering activities or require precautions depending on zone and site. cite6cab0fe95637e55fa4e3e036c1c97d001a37c40b51c542df3c2c4b4edfeba1ff The April 2026 ALTIS enquiry is a no-record/no-known-finding indication, not a contamination clearance. The historic 2006 “no suspicion” certificate is not a current KMRD clearance; property-specific ordnance information requires precise cadastral/site information. cite49c0d84b192b9acf92d0dbc163712ec737cece979b0017414e51241bfed3702d

**Decision consequence:** Obtain parcel-level stormwater extracts, current Altlastenauskunft, KMRD evaluation, intrusive ground/groundwater testing where warranted, formal archaeology response, protection-zone requirements, and binding insurance terms including exclusions, deductibles, sublimits, claims history, and business interruption cover.

### 5. Energy, carbon and transition — **Unknown**

Actual energy and GHG performance cannot be established from the gross volume of **24,499.12 m³**, inspection intervals, or building use alone. Meter topology, 36 months of utility data, fuels, tariffs, BMS trends, refrigerants, certificates, conditioned area, operating hours, and tenant-control allocation are absent.

The archived comparison framework provides courthouse benchmarks of 90 kWh/(m²·a) heat and 20 electricity for buildings up to 3,500 m² NGF, and 70 heat and 25 electricity above 3,500 m² NGF; office benchmarks vary materially with ventilation and conditioning. These are comparison values, not measured performance or retrofit targets.

**Decision consequence:** Do not label the asset efficient or inefficient, or price transition CapEx, without NGF-based energy data, plant/fabric surveys, electrical-capacity analysis, legal applicability review, and an options appraisal covering cost, savings, disruption, incentives, and carbon.

### 6. Location, demand, market, valuation and exit — **Inference / unknown**

The operating courthouse location, built-up setting, institutional occupation, documented rent history, and conditional broad use wording are positive indicators. They do not establish market rent, alternative-use value, exit yield, liquidity, buyer depth, vacancy, or reletting costs.

No verified submarket statistics, comparable transactions, courthouse/secure-office evidence, valuation report, market-rent analysis, buyer evidence, or exit study was supplied. The stated rent therefore cannot be treated as market income, and the property cannot be assigned a supported market value or liquidity conclusion.

**Decision consequence:** Obtain an independent valuation supported by current achieved and asking rents, yields, vacancy, comparable institutional assets, buyer depth, tenant/covenant analysis, CapEx, insurability, and alternative-use constraints. Until then, model longer marketing, a narrower buyer pool, yield expansion, income haircut, delayed exit, and reletting/repurposing costs as sensitivities—not as confirmed impairments.

### 7. Finance, debt and macro transmission — **Unknown**

The actual debt instrument, lender, balance, interest rate, amortisation, maturity, covenant package, hedging, recourse, liquidity, and refinancing terms are unknown. The Eurohypo land charge cannot be used as a proxy for current debt, particularly given the inconsistent amounts of **€595,375,000 and €595,375**.

The macro framework is clear: the ECB sets euro-area key rates, with the cited official table recording effective 17 June 2026 deposit 2.25%, main refinancing 2.40%, and marginal lending 2.65%.[citation:6f6d3762aec6b3ed9fa9c2b6bec1a7bd06587a564cde1868456918876337fecd] Eurostat publishes the HICP as a comparable inflation measure.[citation:1f5511a1994dc4669da4df69d2a90d9c8c8c9f46ba39e8e7679d314b706c2526] These general data do not resolve the lease’s specific German CPI clause or the property’s financing exposure.

**Decision consequence:** Do not approve leverage, distributions, DSCR, LTV, refinancing, or lender-consent assumptions. Obtain facility and security documents, lender statement, payoff/balance, covenants, hedge schedule, guarantees, valuation, and an integrated downside model.

### 8. External dependencies, geopolitics, trade and supply chains — **Supported for identified dependencies; broader exposure unknown**

The immediate external dependency is operational rather than demonstrably geopolitical:

1. FMC maintenance does not transfer;
2. the Chemicon tender recommendation is not a construction award;
3. ptd’s appointment is architectural LP8 only;
4. fire-water, drainage, canal-right, water-protection, and authority interfaces remain unresolved;
5. HPPVO/LBIH and escape-plan evidence is incomplete.

The three-bid process supports competitive tender participation, but not supplier resilience, lead time, substitution capacity, material availability, or contract enforceability. No evidence establishes country, route, sanctions, commodity, or single-source exposure.

**Decision consequence:** Require maintenance continuity, construction-award and programme evidence, contractor capacity and warranty checks, current authority confirmations, and technical alternatives. Do not interpret absence of identified geopolitical exposure as evidence that supply-chain exposure is low.

## Cross-domain transmission

The principal causal chains are:

1. **Title and use → income → value and finance**  
   Parcel, charge, easement, and Hesse-use reconciliation determines lawful ownership, possession, use, lease enforceability, collateral validity, and saleability. This chain is broken at current title instruments and lender/security evidence.

2. **Lease and indexation → gross income → NOI → debt service and value**  
   The documented rent history produces annualised stated rent of €939,062.04, but the link to collectible NOI is broken by missing payment, tenant, security, recoveries, landlord-cost, term, and written-form evidence.

3. **Condition and fire safety → works → disruption → income and exit**  
   Open or undocumented safety items may require works, access restrictions, temporary controls, and operational disruption. The link is broken by absent current inspections, accepted close-outs, programme, and liability allocation.

4. **Garage tender → CapEx and parking disruption → liquidity and valuation**  
   The tender supports an indicative base cost only. Scope, award, timing, parking closure, temporary parking, compensation, and cost allocation remain unknown.

5. **Groundwater, water protection and drainage → permits/design → cost and insurability**  
   These may constrain garage, drainage, waterproofing, and energy works and may affect insurance and delay. Parcel-specific permissions, hydraulic verification, intrusive testing, and policy terms are missing.

6. **Energy data → transition CapEx/Opex → value**  
   No reliable performance series means energy cost, emissions, savings, regulatory exposure, and retrofit economics cannot yet be connected to valuation or financing.

7. **External suppliers and authorities → delivery programme → CapEx and exit**  
   No evidenced geopolitical pathway is established. Contractor availability, maintenance continuity, municipal interfaces, and approvals remain practical delivery risks.

## Contradictions and competing readings

| Issue | Reading supported by the record | Competing reading and consequence | Missing fact that resolves it |
|---|---|---|---|
| Garage works | Three close bids indicate a likely material refurbishment need, with an indicative base around €703k net. | The work may already have been awarded, superseded, or not required as assumed. Without the award and scope, price, timing, liability, and disruption remain unknown. | Executed construction contract, final scope, programme, contingency, and owner/tenant allocation. |
| Maintenance coverage | FMC currently reportedly covers owner maintenance. | Because it will not transfer, support may cease or create landlord leakage at closing. | Novation, replacement contract, service budget, and transition plan. |
| Fire safety | Dated defects and missing evidence indicate a potential backlog. | Defects may have been remediated and only documentary evidence is missing. | Current independent inspections, accepted close-outs, updated plans, and authority/LBIH/HPPVO evidence. |
| Parking | The requirement consistently totals 89 spaces. | The different underground/outdoor allocations may affect legal entitlement, operation, and compensation. | Current approved parking plan, operative agreement, title/easement evidence, and municipal account statement. |
| Drainage | The 33.52/33.62 l/s difference may be immaterial. | It may indicate an unresolved hydraulic or easement basis affecting works and water risk. | Approved hydraulic calculation, current capacity confirmation, and canal-right instrument. |
| Title and land charge | Inconsistencies may be transcription or historical-record errors. | They may reflect an unreleased security, parcel mismatch, or ownership/priority issue. | Certified Grundbuch/Grundakten, cadastral history, charge release, and corporate authority. |
| Environmental status | ALTIS and historic no-suspicion evidence are supportive. | No-record findings do not exclude contamination, groundwater, ordnance, or archaeology. | Current Altlastenauskunft, intrusive assessment, KMRD response, and formal archaeology position. |
| Use flexibility | §34 BauGB and conditional easement wording support continuity and optionality. | Alternative use may require consent and new approvals and may be constrained by water, fire, title, or archaeology. | Enforceable easement instrument, current permits, municipal position, and authority consents. |
| Rent growth | The rent increases may be valid CPI indexation. | Notices or written-form/indexation defects, concessions, side letters, or payment problems may undermine durability. | Executed lease chain, CPI calculations, notices/delivery, payment ledger, security, and side-letter confirmation. |
| Debt charge | Eurohypo may have secured material financing. | The charge may be historic, released, portfolio-related, or a transcription error. | Current land register, facility agreement, lender balance, payoff letter, and release/consent evidence. |

## Conditions precedent and decision gates

Before final investment or financing approval, require:

1. Certified Grundbuch, Grundakten, cadastral extracts, parcel-history reconciliation, easements, canal-rights, charge releases, and corporate authority.
2. Executed lease and amendments, written-form review, CPI calculations and notices, payment ledger, security, tenant identity and financial information, recoveries, service-charge reconciliations, and parking documentation.
3. Current independent condition survey covering structure, envelope, MEP, electrical, ventilation, CO, fire systems, controls, drainage, waterproofing, utilities, and accessibility.
4. Current fire-door, detector, escape-plan, HPPVO/LBIH, fire-water, and authority close-out evidence.
5. Garage works confirmation: executed award or re-tender, scope inclusions/exclusions, programme, contingency, warranties, insurance, bonds, downtime, parking plan, temporary arrangements, and allocation of cost.
6. FMC novation or replacement maintenance contract with budget and no service gap.
7. Current planning/building permits and approved plans, water-protection requirements, archaeological response, drainage capacity, and any contribution or parking-charge confirmation.
8. Parcel-specific hazard, contamination, groundwater, ordnance and insurability evidence, including current policy, exclusions, deductibles, limits, claims history, reinstatement value, and business-interruption basis.
9. Thirty-six months of reconciled utility and meter data, NGF schedule, energy certificate, BMS data, plant/fabric survey, and transition options appraisal.
10. Executed debt/security documents, lender statement, current balance, maturity, rates, hedging, covenants, guarantees, payoff mechanics, and lender consent.
11. Independent valuation using current market evidence, contractual-income verification, CapEx, insurability, tenant durability, buyer depth, and exit sensitivities.

## `assert`

- **Assert that the correct current decision is a conditional diligence hold, not unconditional approval.**
- **Assert the documented rent only as provisional gross contractual rent of €78,255.17 per month / €939,062.04 per year**, pending legal and cash-collection verification.
- **Assert an indicative garage refurbishment base cost of approximately €702,668 net**, subject to scope, award, contingency, fees, tax, escalation, and disruption.
- **Assert that maintenance continuity is not assured after transfer** unless FMC is novated or replaced.
- **Assert that title, use rights, debt, current compliance, energy performance, insurance, valuation, and liquidity remain unresolved.**
- **Assert that physical-risk pathways are material but not evidence of an actual loss event.**
- **Assert that agreement between the domain reports is shared-source corroboration, not independent confirmation.**

## `caveat`

- The property-file facts are generally supplied data-room inputs and are not independently verified unless expressly supported by cited public or official material.
- Historic design records, old inspection reports, database no-findings, generic benchmarks, and city-wide hazard models cannot be treated as current property-specific certificates or clearances.
- The €703k garage figure is one indicative CapEx item and must not be duplicated across models. Maintenance replacement, technical remediation, parking disruption, insurance, financing carry, and exit effects should be modelled as linked but distinct consequences.
- Vacancy, collection loss, invalid indexation, unrecovered costs, and tenant disruption should not all be applied at full severity without an explicit simultaneous-stress case.
- Groundwater, drainage, climate, contamination, ordnance, archaeology, energy, transition, insurance, counterparty resilience, refinancing terms, and alternative-use value should remain **unknown** until property-specific evidence exists.
- The cited ECB and Eurostat material establishes general macro context, not the property’s actual rate exposure or lease-indexation validity.
- The absence of evidence for a geopolitical or trade pathway is an **unknown**, not evidence of no exposure.

## `avoid`

- **Avoid** approving acquisition, refinancing, leverage, distributions, or a clean valuation conclusion before the conditions precedent are satisfied.
- **Avoid** treating €939,062.04 as NOI, durable low-risk income, or market rent.
- **Avoid** relying on the Eurohypo land-charge amount as current debt or collateral exposure.
- **Avoid** treating the Chemicon recommendation or ptd LP8 appointment as proof of a construction award.
- **Avoid** assuming FMC maintenance coverage transfers on sale.
- **Avoid** treating historic fire-water requirements or dated inspection reports as current compliance.
- **Avoid** treating the 89-space total as resolving the conflicting allocation or parking liability.
- **Avoid** converting ALTIS/no-suspicion responses into contamination, ordnance, groundwater, or archaeological clearance.
- **Avoid** using gross volume or generic courthouse benchmarks to infer energy efficiency, savings, or retrofit returns.
- **Avoid** using generic Bad Homburg averages or unsupported yield assumptions to establish value or insurance cost.
- **Avoid** double-counting garage CapEx, technical backlog, groundwater risk, parking disruption, income downside, rate stress, or yield expansion.
- **Avoid** describing the present record as demonstrating low risk merely because multiple reports agree; the reports share the same model, runtime, and principal source materials.